# Multiple Sequence Alignment: From Theory to Practice

Multiple Sequence Alignment (MSA) is one of the most fundamental techniques in bioinformatics. While pairwise alignment compares two sequences, MSA simultaneously aligns three or more sequences to reveal conserved regions, functional domains, and evolutionary relationships that pairwise comparisons would miss.

---

## Learning Objectives

- Understand why MSA is essential and what biological questions it answers
- Learn the progressive alignment strategy and its algorithmic foundations
- Score MSA quality using sum-of-pairs and column-based metrics
- Use major MSA tools (ClustalW, MUSCLE, MAFFT, T-Coffee)
- Read, write, and manipulate alignments with BioPython AlignIO
- Compute consensus sequences and conservation scores
- Build and interpret sequence logos
- Visualize alignments programmatically

## Why this notebook matters

Multiple sequence alignment (MSA) underpins most comparative analyses: phylogenetic tree building, identification of conserved functional residues, domain boundary prediction, and evolutionary rate analysis. Understanding MSA goes beyond running tools — you need to know when progressive alignment (ClustalW, MUSCLE) is good enough versus when you need iterative refinement (MAFFT --genafpair, T-Coffee), how to assess alignment quality, and how to interpret conservation patterns and sequence logos.

## How to use this notebook

1. Run cells top-to-bottom in order — later cells depend on earlier ones
2. Run the environment check cell first to confirm all imports work
3. Read the explanatory text before each code cell — the context matters
4. The exercises at the end are designed to be done after reading each section
5. If a code cell requires internet access (Entrez, PDB download), it is marked — these can be skipped if offline

## Complicated moments explained

- **Progressive alignment errors propagate**: In progressive alignment, once two sequences are aligned incorrectly in an early step, that error is locked in for all subsequent alignments. This is the main weakness of ClustalW and MUSCLE default mode. MAFFT's iterative refinement and T-Coffee's library approach help mitigate this.
- **Tool selection by dataset size**: ClustalW is slow for large datasets (guide tree computation is O(N²)); MUSCLE is fast and accurate up to ~1,000 sequences; MAFFT with `--auto` scales to tens of thousands. For >100,000 sequences, use MAFFT `--parttree` or Clustal Omega. T-Coffee produces highest quality but is too slow beyond ~500 sequences.
- **Gap treatment is critical**: Columns with >50% gaps are often unreliable and should be masked before phylogenetic analysis. Tools like trimAl and Gblocks remove poorly aligned columns.
- **Sum-of-Pairs (SP) score vs. Column Score (CS)**: SP score rewards each correctly aligned pair of residues; CS rewards only columns where all pairs are correct. For benchmarking, CS is more stringent. Real alignments are evaluated against reference alignments from databases like BAliBASE and HOMSTRAD.
- **Amino acid vs. nucleotide alignment**: For protein-coding genes, align protein sequences and then back-translate to codons (codon-aware alignment). Direct nucleotide alignment of coding regions is often misleading because synonymous substitutions saturate at the third codon position.

## Environment check (run this first)

In [ ]:
# Environment check
import numpy as np
import matplotlib.pyplot as plt
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment, PairwiseAligner, substitution_matrices
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import io, subprocess, shutil

print("Core imports ready.")

# Check which external MSA tools are installed
for tool in ['mafft', 'muscle', 'clustalw', 'clustalo']:
    found = shutil.which(tool)
    status = f"found at {found}" if found else "NOT FOUND (install via conda/brew)"
    print(f"  {tool}: {status}")

print("\nNote: if no external MSA tools are found, the notebook will use")
print("a built-in progressive alignment implementation for demonstrations.")

In [ ]:
def exact_msa_three_sequences(seq1, seq2, seq3, match=1, mismatch=-1, gap=-2):
    """
    Exact MSA for three short sequences via 3D dynamic programming.
    Only practical for very short sequences (demonstration purposes).
    """
    n1, n2, n3 = len(seq1), len(seq2), len(seq3)
    
    def score(a, b):
        if a == '-' or b == '-':
            return gap
        return match if a == b else mismatch
    
    # Initialize 3D DP matrix
    dp = np.full((n1 + 1, n2 + 1, n3 + 1), -np.inf)
    dp[0, 0, 0] = 0
    
    # Fill borders
    for i in range(1, n1 + 1):
        dp[i, 0, 0] = dp[i-1, 0, 0] + 2 * gap
    for j in range(1, n2 + 1):
        dp[0, j, 0] = dp[0, j-1, 0] + 2 * gap
    for k in range(1, n3 + 1):
        dp[0, 0, k] = dp[0, 0, k-1] + 2 * gap
    
    # Fill edges
    for i in range(1, n1 + 1):
        for j in range(1, n2 + 1):
            dp[i, j, 0] = max(
                dp[i-1, j-1, 0] + score(seq1[i-1], seq2[j-1]) + 2 * gap,
                dp[i-1, j, 0] + 2 * gap,
                dp[i, j-1, 0] + 2 * gap
            )
    for i in range(1, n1 + 1):
        for k in range(1, n3 + 1):
            dp[i, 0, k] = max(
                dp[i-1, 0, k-1] + score(seq1[i-1], seq3[k-1]) + 2 * gap,
                dp[i-1, 0, k] + 2 * gap,
                dp[i, 0, k-1] + 2 * gap
            )
    for j in range(1, n2 + 1):
        for k in range(1, n3 + 1):
            dp[0, j, k] = max(
                dp[0, j-1, k-1] + score(seq2[j-1], seq3[k-1]) + 2 * gap,
                dp[0, j-1, k] + 2 * gap,
                dp[0, j, k-1] + 2 * gap
            )
    
    # Fill 3D matrix
    for i in range(1, n1 + 1):
        for j in range(1, n2 + 1):
            for k in range(1, n3 + 1):
                dp[i, j, k] = max(
                    dp[i-1, j-1, k-1] + score(seq1[i-1], seq2[j-1]) + score(seq1[i-1], seq3[k-1]) + score(seq2[j-1], seq3[k-1]),
                    dp[i, j-1, k-1] + score(seq2[j-1], seq3[k-1]) + gap,
                    dp[i-1, j, k-1] + score(seq1[i-1], seq3[k-1]) + gap,
                    dp[i-1, j-1, k] + score(seq1[i-1], seq2[j-1]) + gap,
                    dp[i, j, k-1] + 2 * gap,
                    dp[i, j-1, k] + 2 * gap,
                    dp[i-1, j, k] + 2 * gap
                )
    
    print(f"3D DP matrix size: {n1+1} x {n2+1} x {n3+1} = {(n1+1)*(n2+1)*(n3+1)} cells")
    print(f"Optimal score: {dp[n1, n2, n3]}")
    return dp[n1, n2, n3]

# Demo with very short sequences
s1 = "ATCG"
s2 = "ATGG"
s3 = "ACCG"

optimal = exact_msa_three_sequences(s1, s2, s3)
print(f"\nFor 3 sequences of length ~4: manageable")
print(f"For 3 sequences of length 100: {101**3:,} cells")
print(f"For 10 sequences of length 100: {101**10:.2e} cells -- impossible!")

---

## 3. Progressive Alignment

Since exact MSA is intractable, practical tools use **heuristic** approaches. The most widely used is **progressive alignment**, pioneered by Feng and Doolittle (1987) and implemented in ClustalW.

### The Progressive Strategy

```
Progressive Alignment Pipeline
================================

Step 1: All-vs-all pairwise alignment
  s1 vs s2, s1 vs s3, ..., s4 vs s5
       |
       v
Step 2: Build distance matrix from pairwise scores
       |                s1    s2    s3    s4    s5
       |          s1  [0.00  0.23  0.45  0.67  0.71]
       |          s2  [0.23  0.00  0.38  0.62  0.65]
       |          ...
       v
Step 3: Build guide tree (UPGMA or NJ)
       |          +----- s1
       |       +--|
       |       |  +----- s3
       |    +--|
       |    |  +-------- s2
       |  --|
       |    |  +-------- s4
       |    +--|
       |       +-------- s5
       v
Step 4: Align sequences following guide tree order
  - Align s1 + s3  (most similar pair)
  - Align (s1,s3) + s2  (alignment to sequence)
  - Align s4 + s5
  - Align (s1,s3,s2) + (s4,s5)  (alignment to alignment)
       |
       v
Step 5: Final MSA
```

### The Chicken-and-Egg Problem

We need a tree to guide the alignment, but we need an alignment to build a good tree. Progressive alignment solves this by:
1. Building a **rough** guide tree from pairwise distances (no MSA needed)
2. Using this guide tree to produce an MSA
3. Optionally iterating: use the MSA to build a better tree, then re-align

In [ ]:
def kmer_distance(seq1, seq2, k=3):
    """
    Alignment-free distance based on shared k-mers.
    Returns fraction of k-mers unique to either sequence.
    """
    kmers1 = set(seq1[i:i+k] for i in range(len(seq1) - k + 1))
    kmers2 = set(seq2[i:i+k] for i in range(len(seq2) - k + 1))
    
    if not kmers1 and not kmers2:
        return 0.0
    
    shared = kmers1 & kmers2
    total = kmers1 | kmers2
    
    return 1.0 - len(shared) / len(total)


def build_distance_matrix(sequences, names, k=3):
    """
    Build a distance matrix from sequences using k-mer distance.
    """
    n = len(sequences)
    matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i + 1, n):
            d = kmer_distance(sequences[i], sequences[j], k)
            matrix[i, j] = matrix[j, i] = d
    
    return matrix


# Example sequences (globin-like)
sequences = {
    'HBA_HUMAN':  'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH',
    'HBA_MOUSE':  'MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH',
    'HBB_HUMAN':  'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST',
    'HBB_MOUSE':  'MVHLTDAEKAAVNGLWGKVNSDEVGGEALGRLLVVYPWTQRYFDSFGDLSS',
    'MYG_HUMAN':  'MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLK',
}

names = list(sequences.keys())
seqs = list(sequences.values())

dm = build_distance_matrix(seqs, names)

print("K-mer distance matrix (k=3):")
print(f"{'':>12}", end='')
for name in names:
    print(f"{name:>12}", end='')
print()

for i, name in enumerate(names):
    print(f"{name:>12}", end='')
    for j in range(len(names)):
        print(f"{dm[i,j]:>12.3f}", end='')
    print()

### 3.1 Building the Guide Tree with UPGMA

UPGMA (Unweighted Pair Group Method with Arithmetic Mean) is the simplest hierarchical clustering method. At each step:

1. Find the closest pair of clusters
2. Merge them into a new cluster
3. Update distances as the arithmetic mean of all pairwise distances
4. Repeat until one cluster remains

UPGMA assumes a molecular clock (equal rates of evolution), which is generally false -- but it is fast and sufficient for building a rough guide tree.

In [ ]:
def upgma(dist_matrix, names):
    """
    UPGMA clustering. Returns a Newick-format tree string.
    """
    n = len(names)
    D = dist_matrix.copy()
    node_names = list(names)
    # Track cluster sizes for weighted averaging
    cluster_sizes = [1] * n
    
    while len(node_names) > 1:
        m = len(node_names)
        
        # Find minimum distance (off-diagonal)
        min_dist = np.inf
        min_i, min_j = 0, 1
        for i in range(m):
            for j in range(i + 1, m):
                if D[i, j] < min_dist:
                    min_dist = D[i, j]
                    min_i, min_j = i, j
        
        i, j = min_i, min_j
        branch_length = min_dist / 2
        
        # Create new node
        new_name = f"({node_names[i]}:{branch_length:.4f},{node_names[j]}:{branch_length:.4f})"
        new_size = cluster_sizes[i] + cluster_sizes[j]
        
        # Compute new distances (arithmetic mean)
        new_D = np.zeros((m - 1, m - 1))
        new_names = []
        new_sizes = []
        idx_map = []
        
        for k in range(m):
            if k != i and k != j:
                new_names.append(node_names[k])
                new_sizes.append(cluster_sizes[k])
                idx_map.append(k)
        new_names.append(new_name)
        new_sizes.append(new_size)
        
        # Copy existing distances
        for a, ka in enumerate(idx_map):
            for b, kb in enumerate(idx_map):
                new_D[a, b] = D[ka, kb]
        
        # Compute distances to new merged cluster
        for a, ka in enumerate(idx_map):
            d_new = (cluster_sizes[i] * D[ka, i] + cluster_sizes[j] * D[ka, j]) / new_size
            new_D[a, -1] = new_D[-1, a] = d_new
        
        D = new_D
        node_names = new_names
        cluster_sizes = new_sizes
    
    return node_names[0] + ";"


guide_tree = upgma(dm, names)
print("Guide tree (Newick):")
print(guide_tree)

### 3.2 Aligning Alignments: The Profile Concept

In progressive alignment, we don't just align single sequences. We also need to:
- Align a sequence to an existing alignment
- Align two alignments together

This is done by converting each alignment column into a **profile** -- a vector of character frequencies. The substitution score between two profile columns is the **average** of all pairwise scores:

$$
S(\text{col}_1, \text{col}_2) = \frac{\sum_{a \in \text{col}_1} \sum_{b \in \text{col}_2} s(a, b)}{|\text{col}_1| \times |\text{col}_2|}
$$

For example, aligning column `[A, C]` against column `[T, G]` with a substitution matrix $m$:

$$
S = \frac{m[A][T] + m[A][G] + m[C][T] + m[C][G]}{2 \times 2}
$$

In [ ]:
from collections import Counter

def make_profile(alignment):
    """
    Convert a list of aligned sequences into a position-specific frequency profile.
    Returns list of Counter objects, one per position.
    """
    if isinstance(alignment, str):
        alignment = [alignment]
    
    length = len(alignment[0])
    profile = []
    
    for pos in range(length):
        col = [seq[pos] for seq in alignment]
        counts = Counter(col)
        # Normalize to frequencies
        total = sum(v for k, v in counts.items() if k != '-')
        freq = {k: v / max(total, 1) for k, v in counts.items()}
        profile.append(freq)
    
    return profile


def profile_score(profile_col1, profile_col2, match=1, mismatch=-1):
    """
    Score alignment of two profile columns using average pairwise scores.
    """
    total_score = 0.0
    total_weight = 0.0
    
    for char1, freq1 in profile_col1.items():
        if char1 == '-':
            continue
        for char2, freq2 in profile_col2.items():
            if char2 == '-':
                continue
            s = match if char1 == char2 else mismatch
            weight = freq1 * freq2
            total_score += s * weight
            total_weight += weight
    
    return total_score / total_weight if total_weight > 0 else 0


# Demonstrate profile construction
aln = ["ATCGA", "ATCGA", "ATTGA"]
prof = make_profile(aln)

print("Alignment:")
for seq in aln:
    print(f"  {seq}")
print("\nProfile (position -> character frequencies):")
for i, col in enumerate(prof):
    chars = ', '.join(f"{k}:{v:.2f}" for k, v in sorted(col.items()) if k != '-')
    print(f"  Position {i}: {chars}")

### 3.3 Progressive MSA Implementation

Let us put together a simplified progressive MSA. We will:
1. Compute all pairwise distances
2. Build a UPGMA guide tree
3. Align pairs and groups following the tree order

In [ ]:
def simple_nw_align(seq1, seq2, match=2, mismatch=-1, gap=-2):
    """
    Simple Needleman-Wunsch global alignment for two sequences.
    Returns aligned sequences as strings.
    """
    n, m = len(seq1), len(seq2)
    dp = np.zeros((n + 1, m + 1))
    
    for i in range(1, n + 1):
        dp[i, 0] = dp[i-1, 0] + gap
    for j in range(1, m + 1):
        dp[0, j] = dp[0, j-1] + gap
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            dp[i, j] = max(
                dp[i-1, j-1] + s,
                dp[i-1, j] + gap,
                dp[i, j-1] + gap
            )
    
    # Traceback
    aln1, aln2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            if dp[i, j] == dp[i-1, j-1] + s:
                aln1.append(seq1[i-1])
                aln2.append(seq2[j-1])
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i, j] == dp[i-1, j] + gap:
            aln1.append(seq1[i-1])
            aln2.append('-')
            i -= 1
        else:
            aln1.append('-')
            aln2.append(seq2[j-1])
            j -= 1
    
    return ''.join(reversed(aln1)), ''.join(reversed(aln2))


def merge_into_alignment(aln_group, new_seq_aligned, ref_aligned):
    """
    When we align a reference from an existing alignment group to a new sequence,
    gaps may be added to the reference. Propagate those gaps to all other
    sequences in the group.
    """
    result = []
    for seq in aln_group:
        new_seq = []
        orig_idx = 0
        for ref_char in ref_aligned:
            if ref_char == '-' and (orig_idx >= len(seq) or (orig_idx < len(seq))):
                # Check if this gap was newly introduced
                new_seq.append('-')
            else:
                if orig_idx < len(seq):
                    new_seq.append(seq[orig_idx])
                else:
                    new_seq.append('-')
                orig_idx += 1
        result.append(''.join(new_seq))
    return result


# Demonstrate pairwise alignment
a1, a2 = simple_nw_align("MVLSPADKTNVKAAWGKVGA", "MVHLTPEEKSAVTALWGKVNV")
print("Pairwise alignment:")
print(f"  {a1}")
print(f"  {a2}")
print(f"  {''.join('|' if a == b else ' ' for a, b in zip(a1, a2))}")

### 3.4 Iterative Refinement

Progressive alignment has a fundamental limitation: **errors made early propagate** through the entire alignment. If the guide tree incorrectly pairs two dissimilar sequences first, that mistake is frozen into the final result.

**Iterative alignment** addresses this:
1. Perform progressive MSA to get an initial alignment
2. Build a new tree from the MSA (using alignment-based distances)
3. Re-align using the new guide tree
4. Repeat until convergence

This is the strategy used by tools like MUSCLE and MAFFT for refinement.

---

## 4. Scoring MSA Quality

How do we know if one MSA is better than another? Several scoring metrics exist.

### 4.1 Sum-of-Pairs (SP) Score

The SP score sums the pairwise alignment scores for every pair of sequences at each column:

$$
SP = \sum_{\text{column}} \sum_{i < j} s(a_i, a_j)
$$

### 4.2 Column Score (CS)

The column score counts the fraction of columns that are aligned identically in a reference alignment.

In [ ]:
def sum_of_pairs_score(alignment, match=1, mismatch=-1, gap_penalty=-2):
    """
    Calculate the Sum-of-Pairs score for an MSA.
    
    For each column, sum the score of every pair of characters.
    """
    n_seqs = len(alignment)
    aln_length = len(alignment[0])
    
    total_score = 0
    column_scores = []
    
    for pos in range(aln_length):
        col_score = 0
        for i in range(n_seqs):
            for j in range(i + 1, n_seqs):
                a, b = alignment[i][pos], alignment[j][pos]
                if a == '-' or b == '-':
                    col_score += gap_penalty
                elif a == b:
                    col_score += match
                else:
                    col_score += mismatch
        column_scores.append(col_score)
        total_score += col_score
    
    return total_score, column_scores


def column_conservation_score(alignment):
    """
    For each column, compute the fraction of the most common non-gap character.
    """
    n_seqs = len(alignment)
    aln_length = len(alignment[0])
    scores = []
    
    for pos in range(aln_length):
        col = [alignment[s][pos] for s in range(n_seqs)]
        counts = Counter(c for c in col if c != '-')
        if counts:
            max_freq = max(counts.values()) / n_seqs
            scores.append(max_freq)
        else:
            scores.append(0.0)
    
    return scores


# Example MSA
msa_example = [
    "MVLSPADKTNVKAAWGKVGA",
    "MVLSGEDKSNIKAAWGKIGG",
    "MVHLTPEEKSAVTALWGKVNV",
    "MVHLTDAEK-AVNGLWGKVNS",
]

# Pad to equal length
max_len = max(len(s) for s in msa_example)
msa_padded = [s.ljust(max_len, '-') for s in msa_example]

sp_total, sp_cols = sum_of_pairs_score(msa_padded)
cons_scores = column_conservation_score(msa_padded)

print(f"Sum-of-Pairs score: {sp_total}")
print(f"Mean column conservation: {np.mean(cons_scores):.3f}")
print(f"Fully conserved columns: {sum(1 for s in cons_scores if s == 1.0)} / {len(cons_scores)}")

# Visualize column conservation
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(cons_scores)), cons_scores, color='steelblue', alpha=0.8)
ax.set_xlabel('Alignment Position')
ax.set_ylabel('Conservation')
ax.set_title('Per-Column Conservation Score')
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Fully conserved')
ax.legend()
plt.tight_layout()
plt.show()

---

## 5. MSA Tools: ClustalW, MUSCLE, MAFFT, T-Coffee

### Comparison of Major MSA Tools

| Tool | Algorithm | Speed | Accuracy | Best For |
|------|-----------|-------|----------|----------|
| **ClustalW** | Progressive (NJ guide tree) | Slow | Moderate | Small datasets (<100), legacy scripts |
| **Clustal Omega** | HMM profile-profile + mBed clustering | Fast | Good | 100–100,000 sequences |
| **MUSCLE v3** | Progressive + iterative refinement | Fast | Good | 100–1,000 sequences |
| **MUSCLE v5** | PPP algorithm + ensemble refinement | Very fast | Excellent | Any size, recommended default |
| **MAFFT** | FFT-based (L-INS-i, G-INS-i) + iterative | Very fast | Excellent | Diverse seqs, genomes, structural RNA |
| **T-Coffee** | Consistency-based library alignment | Slow | Highest | Small highly divergent datasets (<500) |
| **DIALIGN** | Local alignments | Moderate | Variable | Mosaic proteins, local motifs |

**When to choose what:**
- **Routine protein MSA, any size**: MAFFT `--auto` or MUSCLE v5 — both are fast and highly accurate
- **RNA structure-aware alignment**: MAFFT `--genafpair` or `--linsi` (considers secondary structure)
- **Highly divergent sequences (<30% identity)**: T-Coffee or MAFFT `--genafpair`
- **Ultra-large (>50,000 seqs)**: MAFFT `--parttree --retree 1` or DIAMOND+clustering pre-step
- **Legacy compatibility**: ClustalW output format is still required by some older tools

**Note on MUSCLE versions**: MUSCLE v5 (released 2021) is a complete rewrite using a new algorithm that is faster and more accurate than v3. On macOS/Linux, `muscle` may refer to either version depending on your conda channel. Check with `muscle -version`.

In [ ]:
import tempfile
import os

# Prepare example FASTA sequences for tool demonstration
globin_sequences = {
    'HBA_HUMAN':   'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH',
    'HBA_MOUSE':   'MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH',
    'HBA_CHICKEN':  'MVLSAADKNNVKGIFTKIAGHAEEYGAETLERMFTTYPPTKTYFPHFDLSH',
    'HBB_HUMAN':   'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST',
    'HBB_MOUSE':   'MVHLTDAEKAAVNGLWGKVNSDEVGGEALGRLLVVYPWTQRYFDSFGDLSS',
    'MYG_HUMAN':   'MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLK',
    'MYG_WHALE':   'MVLSDAEWQLVLNIWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLK',
}

# Write FASTA file
fasta_path = os.path.join(tempfile.gettempdir(), 'globins.fasta')
with open(fasta_path, 'w') as f:
    for name, seq in globin_sequences.items():
        f.write(f'>{name}\n{seq}\n')

print(f"FASTA file written to: {fasta_path}")
print(f"Contains {len(globin_sequences)} sequences")
print()

# Display the FASTA content
for name, seq in globin_sequences.items():
    print(f">{name}")
    print(f"{seq}")

In [ ]:
import subprocess
import shutil

def run_msa_tool(tool_name, input_fasta, output_path):
    """
    Try to run an MSA tool. Returns True if successful, False otherwise.
    """
    commands = {
        'muscle': ['muscle', '-in', input_fasta, '-out', output_path],
        'mafft': ['mafft', '--auto', input_fasta],
        'clustalo': ['clustalo', '-i', input_fasta, '-o', output_path, '--outfmt=fasta', '--force'],
    }
    
    if tool_name not in commands:
        print(f"Unknown tool: {tool_name}")
        return False
    
    if not shutil.which(commands[tool_name][0]):
        print(f"{tool_name} not found in PATH. Install with: conda install -c bioconda {tool_name}")
        return False
    
    try:
        if tool_name == 'mafft':
            result = subprocess.run(commands[tool_name], capture_output=True, text=True, timeout=60)
            with open(output_path, 'w') as f:
                f.write(result.stdout)
        else:
            subprocess.run(commands[tool_name], capture_output=True, text=True, timeout=60)
        print(f"{tool_name} completed successfully.")
        return True
    except (subprocess.TimeoutExpired, FileNotFoundError) as e:
        print(f"{tool_name} failed: {e}")
        return False


# Try each available tool
for tool in ['muscle', 'mafft', 'clustalo']:
    out_path = os.path.join(tempfile.gettempdir(), f'globins_{tool}.fasta')
    success = run_msa_tool(tool, fasta_path, out_path)
    if success and os.path.exists(out_path):
        print(f"  Output: {out_path}")
    print()

---

## 6. BioPython AlignIO: Reading, Writing, and Manipulating MSAs

BioPython's `AlignIO` module provides a unified interface for working with alignment files in many formats.

In [ ]:
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import io

# Create an alignment object directly
records = [
    SeqRecord(Seq("MVLSPADKTNVKAAWGKVGA-HAGEYGAEALERMFLSFPTTKTYFPHFDLSH"), id="HBA_HUMAN"),
    SeqRecord(Seq("MVLSGEDKSNIKAAWGKIGG-HGAEYGAEALERMFASFPTTKTYFPHFDVSH"), id="HBA_MOUSE"),
    SeqRecord(Seq("MVLSAADKNNVKGIFTKIAG-HAEEYGAETLERMFTTYPPTKTYFPHFDLSH"), id="HBA_CHICKEN"),
    SeqRecord(Seq("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST-"), id="HBB_HUMAN"),
    SeqRecord(Seq("MVHLTDAEKAAVNGLWGKVNSDEVGGEALGRLLVVYPWTQRYFDSFGDLSS-"), id="HBB_MOUSE"),
]

alignment = MultipleSeqAlignment(records)

print(f"Alignment: {len(alignment)} sequences, {alignment.get_alignment_length()} columns")
print()

# Display alignment
for record in alignment:
    print(f"{record.id:>12}  {str(record.seq)}")

In [ ]:
# Access individual columns and sequences
print("Column slicing:")
print(f"  Column 0: {alignment[:, 0]}")
print(f"  Column 5: {alignment[:, 5]}")
print(f"  Columns 0-9: ")
for record in alignment:
    print(f"    {record.id:>12}  {str(record.seq[:10])}")

print("\nSequence slicing:")
print(f"  First sequence: {alignment[0].id} = {str(alignment[0].seq[:30])}...")
print(f"  Last sequence:  {alignment[-1].id} = {str(alignment[-1].seq[:30])}...")

# Sub-alignment
sub_aln = alignment[:3, 10:30]  # First 3 sequences, positions 10-30
print(f"\nSub-alignment (3 seqs, pos 10-30):")
for record in sub_aln:
    print(f"  {record.id:>12}  {str(record.seq)}")

In [ ]:
# Write alignment in different formats
formats = ['fasta', 'clustal', 'phylip', 'stockholm']

for fmt in formats:
    output = io.StringIO()
    try:
        AlignIO.write(alignment, output, fmt)
        content = output.getvalue()
        # Show first few lines
        lines = content.strip().split('\n')
        print(f"=== {fmt.upper()} format ===")
        for line in lines[:6]:
            print(f"  {line}")
        if len(lines) > 6:
            print(f"  ... ({len(lines)} total lines)")
        print()
    except Exception as e:
        print(f"=== {fmt.upper()} format: Error - {e} ===")
        print()

---

## 7. Consensus Sequences and Conservation Scores

A **consensus sequence** represents the most common character at each position. It condenses the information from many sequences into one representative.

Types of consensus:
- **Strict consensus**: only positions where ALL sequences agree (rest become 'X' or '?')
- **Majority consensus**: most common character at each position (with a threshold)
- **IUPAC consensus**: uses ambiguity codes (e.g., R = A or G, Y = C or T)

In [ ]:
def compute_consensus(alignment, threshold=0.5, gap_char='-'):
    """
    Compute consensus sequence from an alignment.
    
    Parameters:
        alignment: list of equal-length strings
        threshold: minimum frequency to be consensus (default 0.5 = majority rule)
        gap_char: character to use for ambiguous positions
    
    Returns:
        consensus string, list of conservation scores
    """
    n_seqs = len(alignment)
    aln_length = len(alignment[0])
    
    consensus = []
    conservation = []
    
    for pos in range(aln_length):
        col = [alignment[s][pos] for s in range(n_seqs)]
        counts = Counter(c for c in col if c != '-')
        
        if not counts:
            consensus.append('-')
            conservation.append(0.0)
            continue
        
        most_common_char, most_common_count = counts.most_common(1)[0]
        freq = most_common_count / n_seqs
        conservation.append(freq)
        
        if freq >= threshold:
            consensus.append(most_common_char)
        else:
            consensus.append('x')  # ambiguous
    
    return ''.join(consensus), conservation


# Apply to our alignment
aln_strings = [str(record.seq) for record in alignment]

consensus_50, cons_scores = compute_consensus(aln_strings, threshold=0.5)
consensus_80, _ = compute_consensus(aln_strings, threshold=0.8)
consensus_100, _ = compute_consensus(aln_strings, threshold=1.0)

print("Alignment:")
for record in alignment:
    print(f"  {record.id:>12}  {str(record.seq)}")
print()
print(f"  {'Consensus50':>12}  {consensus_50}")
print(f"  {'Consensus80':>12}  {consensus_80}")
print(f"  {'Strict':>12}  {consensus_100}")
print()
print(f"Fully conserved positions (100%): {sum(1 for s in cons_scores if s >= 1.0)}")
print(f"Highly conserved (>=80%): {sum(1 for s in cons_scores if s >= 0.8)}")
print(f"Total positions: {len(cons_scores)}")

In [ ]:
from Bio.Align import AlignInfo

# BioPython's built-in consensus
summary = AlignInfo.SummaryInfo(alignment)

# Dumb consensus (simple majority)
dumb_cons = summary.dumb_consensus(threshold=0.5, ambiguous='X')
print(f"BioPython dumb_consensus (50%): {dumb_cons}")

# Position-Specific Scoring Matrix (PSSM)
pssm = summary.pos_specific_score_matrix()
print("\nPosition-Specific Scoring Matrix (first 10 positions):")
print(f"{'Pos':>4}", end='')
for aa in sorted(pssm[0].keys()):
    if aa != '-':
        print(f"{aa:>5}", end='')
print()

for i in range(min(10, len(pssm))):
    print(f"{i:>4}", end='')
    for aa in sorted(pssm[i].keys()):
        if aa != '-':
            print(f"{pssm[i][aa]:>5.0f}", end='')
    print()

---

## 8. Information Content and Sequence Logos

### Information Theory in MSA

A **sequence logo** visualizes conservation at each position using **information content** measured in bits. At each position:

$$
R_i = \log_2(N) - H_i = \log_2(N) + \sum_{a} f_{a,i} \log_2(f_{a,i})
$$

Where:
- $N$ = alphabet size (4 for DNA, 20 for protein)
- $H_i$ = Shannon entropy at position $i$
- $f_{a,i}$ = frequency of character $a$ at position $i$

Maximum information: $\log_2(4) = 2$ bits for DNA, $\log_2(20) \approx 4.32$ bits for protein.

The height of each letter in the logo is proportional to $f_{a,i} \times R_i$.

In [ ]:
import math

def information_content(alignment, alphabet_size=20):
    """
    Calculate per-position information content for an alignment.
    
    Returns:
        positions: list of dicts {char: height} for each position
        total_ic: list of total information content per position
    """
    n_seqs = len(alignment)
    aln_length = len(alignment[0])
    max_entropy = math.log2(alphabet_size)
    
    positions = []
    total_ic = []
    
    for pos in range(aln_length):
        col = [alignment[s][pos] for s in range(n_seqs)]
        counts = Counter(c for c in col if c != '-')
        total = sum(counts.values())
        
        if total == 0:
            positions.append({})
            total_ic.append(0.0)
            continue
        
        # Calculate entropy
        entropy = 0.0
        freqs = {}
        for char, count in counts.items():
            f = count / total
            freqs[char] = f
            if f > 0:
                entropy -= f * math.log2(f)
        
        # Information content
        ic = max_entropy - entropy
        total_ic.append(ic)
        
        # Letter heights
        heights = {char: freq * ic for char, freq in freqs.items()}
        positions.append(heights)
    
    return positions, total_ic


# Calculate for our globin alignment
pos_data, ic_values = information_content(aln_strings, alphabet_size=20)

print("Information content per position (bits):")
print(f"Max possible (protein): {math.log2(20):.2f} bits")
print(f"Mean IC: {np.mean(ic_values):.2f} bits")
print()
print("Top 10 most conserved positions:")
sorted_positions = sorted(enumerate(ic_values), key=lambda x: -x[1])
for pos, ic in sorted_positions[:10]:
    col = [aln_strings[s][pos] for s in range(len(aln_strings))]
    counts = Counter(c for c in col if c != '-')
    print(f"  Position {pos:>3}: IC={ic:.2f} bits  Column={''.join(col)}  ({dict(counts)})")

In [ ]:
def plot_sequence_logo(positions, total_ic, start=0, end=None, title="Sequence Logo"):
    """
    Plot a simplified sequence logo using matplotlib.
    Each position shows stacked letters scaled by information content.
    """
    if end is None:
        end = len(positions)
    
    positions = positions[start:end]
    total_ic_slice = total_ic[start:end]
    n_pos = len(positions)
    
    # Color scheme for amino acids
    aa_colors = {
        # Hydrophobic: green
        'A': '#33cc33', 'V': '#33cc33', 'I': '#33cc33', 'L': '#33cc33',
        'M': '#33cc33', 'F': '#33cc33', 'W': '#33cc33', 'P': '#33cc33',
        # Polar: blue
        'S': '#3366ff', 'T': '#3366ff', 'N': '#3366ff', 'Q': '#3366ff',
        'Y': '#3366ff', 'C': '#3366ff', 'G': '#3366ff',
        # Positive: red
        'K': '#cc0000', 'R': '#cc0000', 'H': '#cc0000',
        # Negative: magenta
        'D': '#cc00cc', 'E': '#cc00cc',
    }
    
    fig, ax = plt.subplots(figsize=(max(n_pos * 0.4, 10), 4))
    
    for i, (pos_heights, ic) in enumerate(zip(positions, total_ic_slice)):
        if not pos_heights:
            continue
        
        # Sort by height (smallest at bottom)
        sorted_chars = sorted(pos_heights.items(), key=lambda x: x[1])
        
        y_offset = 0
        for char, height in sorted_chars:
            if height > 0.01:  # Skip negligible contributions
                color = aa_colors.get(char, '#888888')
                ax.bar(i, height, bottom=y_offset, width=0.9,
                       color=color, edgecolor='none', alpha=0.9)
                if height > 0.15:  # Only label if tall enough
                    ax.text(i, y_offset + height/2, char,
                            ha='center', va='center', fontsize=8,
                            fontweight='bold', color='white')
                y_offset += height
    
    ax.set_xlabel('Position', fontsize=11)
    ax.set_ylabel('Information Content (bits)', fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.set_xlim(-0.5, n_pos - 0.5)
    ax.set_ylim(0, math.log2(20) + 0.2)
    ax.axhline(y=math.log2(20), color='gray', linestyle=':', alpha=0.3)
    
    # Add position labels
    if n_pos <= 60:
        ax.set_xticks(range(n_pos))
        ax.set_xticklabels(range(start, end), fontsize=7, rotation=90)
    
    plt.tight_layout()
    plt.show()


# Plot sequence logo for our globin alignment
plot_sequence_logo(pos_data, ic_values, start=0, end=51,
                   title="Globin Family Alignment - Sequence Logo")

---

## 9. Alignment Visualization

Good visualization is essential for interpreting MSAs. Professional tools like **JalView** provide interactive viewing, but we can create useful visualizations programmatically.

In [ ]:
def plot_alignment_heatmap(alignment, names, title="Multiple Sequence Alignment"):
    """
    Visualize an MSA as a colored heatmap.
    Colors represent amino acid properties.
    """
    # Amino acid property groups
    aa_groups = {
        'hydrophobic': set('AILMFWVP'),
        'polar': set('STYCNQG'),
        'positive': set('KRH'),
        'negative': set('DE'),
        'gap': set('-'),
    }
    
    group_colors = {
        'hydrophobic': 0.2,
        'polar': 0.5,
        'positive': 0.8,
        'negative': 0.95,
        'gap': 0.0,
    }
    
    n_seqs = len(alignment)
    aln_length = len(alignment[0])
    
    # Create numeric matrix
    matrix = np.zeros((n_seqs, aln_length))
    for i in range(n_seqs):
        for j in range(aln_length):
            char = alignment[i][j].upper()
            for group, chars in aa_groups.items():
                if char in chars:
                    matrix[i, j] = group_colors[group]
                    break
    
    fig, ax = plt.subplots(figsize=(min(aln_length * 0.3, 20), max(n_seqs * 0.5, 3)))
    
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlBu_r', vmin=0, vmax=1)
    
    # Add sequence characters
    if aln_length <= 60:
        for i in range(n_seqs):
            for j in range(aln_length):
                ax.text(j, i, alignment[i][j], ha='center', va='center',
                        fontsize=6, fontweight='bold')
    
    ax.set_yticks(range(n_seqs))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Alignment Position', fontsize=11)
    ax.set_title(title, fontsize=13)
    
    plt.tight_layout()
    plt.show()


aln_names = [record.id for record in alignment]
aln_seqs = [str(record.seq) for record in alignment]

plot_alignment_heatmap(aln_seqs, aln_names, title="Globin Alignment")

In [ ]:
def plot_identity_matrix(alignment, names):
    """
    Plot pairwise percent identity matrix from an MSA.
    """
    n = len(alignment)
    identity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            matches = sum(1 for a, b in zip(alignment[i], alignment[j])
                         if a == b and a != '-')
            aligned = sum(1 for a, b in zip(alignment[i], alignment[j])
                         if a != '-' and b != '-')
            identity_matrix[i, j] = 100 * matches / aligned if aligned > 0 else 0
    
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(identity_matrix, cmap='YlOrRd', vmin=0, vmax=100)
    
    ax.set_xticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n))
    ax.set_yticklabels(names, fontsize=9)
    
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{identity_matrix[i,j]:.0f}",
                    ha='center', va='center', fontsize=9,
                    color='white' if identity_matrix[i,j] > 60 else 'black')
    
    plt.colorbar(im, label='% Identity')
    ax.set_title('Pairwise Identity Matrix', fontsize=13)
    plt.tight_layout()
    plt.show()


plot_identity_matrix(aln_seqs, aln_names)

---

## 10. Finding Conserved Motifs and Functional Regions

MSAs reveal conserved motifs -- short patterns that are preserved across evolution. These often correspond to:
- Active sites of enzymes
- DNA/RNA binding domains
- Protein-protein interaction interfaces
- Post-translational modification sites

In [ ]:
def find_conserved_blocks(alignment, min_conservation=0.8, min_length=3):
    """
    Find blocks of consecutive positions with high conservation.
    
    Parameters:
        alignment: list of aligned sequences
        min_conservation: minimum fraction of identical residues
        min_length: minimum block length
    
    Returns:
        List of (start, end, mean_conservation) tuples
    """
    n_seqs = len(alignment)
    aln_length = len(alignment[0])
    
    # Calculate per-position conservation
    conservation = []
    for pos in range(aln_length):
        col = [alignment[s][pos] for s in range(n_seqs)]
        counts = Counter(c for c in col if c != '-')
        if counts:
            max_freq = max(counts.values()) / n_seqs
            conservation.append(max_freq)
        else:
            conservation.append(0.0)
    
    # Find blocks
    blocks = []
    in_block = False
    block_start = 0
    
    for pos in range(aln_length):
        if conservation[pos] >= min_conservation:
            if not in_block:
                block_start = pos
                in_block = True
        else:
            if in_block:
                block_length = pos - block_start
                if block_length >= min_length:
                    mean_cons = np.mean(conservation[block_start:pos])
                    blocks.append((block_start, pos, mean_cons))
                in_block = False
    
    # Handle block at end
    if in_block:
        block_length = aln_length - block_start
        if block_length >= min_length:
            mean_cons = np.mean(conservation[block_start:aln_length])
            blocks.append((block_start, aln_length, mean_cons))
    
    return blocks, conservation


blocks, conservation = find_conserved_blocks(aln_seqs, min_conservation=0.8, min_length=3)

print(f"Found {len(blocks)} conserved blocks (>= 80% conservation, >= 3 residues):")
print()
for start, end, mean_cons in blocks:
    print(f"Block [{start}-{end}] (length {end-start}, mean conservation: {mean_cons:.2f})")
    for i, record in enumerate(alignment):
        block_seq = str(record.seq[start:end])
        print(f"  {record.id:>12}  {block_seq}")
    print()

In [ ]:
def plot_conservation_with_blocks(conservation, blocks, title="Conservation Profile"):
    """
    Plot conservation scores with conserved blocks highlighted.
    """
    fig, ax = plt.subplots(figsize=(14, 4))
    
    ax.bar(range(len(conservation)), conservation, color='steelblue', alpha=0.7, width=1.0)
    
    # Highlight conserved blocks
    for start, end, mean_cons in blocks:
        ax.axvspan(start - 0.5, end - 0.5, alpha=0.2, color='red',
                   label=f'Block {start}-{end}' if start == blocks[0][0] else '')
    
    ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Conservation threshold')
    ax.set_xlabel('Alignment Position', fontsize=11)
    ax.set_ylabel('Conservation Score', fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


plot_conservation_with_blocks(conservation, blocks,
                              title="Globin Alignment - Conservation with Conserved Blocks")

---

## 11. Real-World Example: Aligning a Protein Family with BioPython

Let us fetch real protein sequences, align them, and analyze the results.

In [ ]:
from Bio import SeqIO

# Create a realistic protein family dataset: Cytochrome c from diverse organisms
cytc_sequences = {
    'CYC_HUMAN':   'MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE',
    'CYC_HORSE':   'MGDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKGERADLIAYLKKATNE',
    'CYC_MOUSE':   'MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKSEREDLIAYLKKATNE',
    'CYC_CHICKEN': 'MGDIEKGKKIFVQKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAEGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKSERVDLIAYLKDATSK',
    'CYC_TUNA':    'MGDVAKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLIGRKTGQAEGFSYTDANKSKGIVWNENILMEYLENPKKYIPGTKMIFAGIKKKGERQDLIAYLKSATS-',
    'CYC_YEAST':   'MTEFKAGSAKKGATLFKTRCLQCHTVEKGGPHKVGPNLHGIFGRHSGQAEGYSYTDANIKKNVLWDENNMSEYLTNPKKYIPGTKMAFGGLKKEKDRNDLITYLKKACE',
}

# Write FASTA
cytc_fasta = os.path.join(tempfile.gettempdir(), 'cytc.fasta')
with open(cytc_fasta, 'w') as f:
    for name, seq in cytc_sequences.items():
        f.write(f'>{name}\n{seq}\n')

print(f"Cytochrome c sequences from {len(cytc_sequences)} organisms:")
for name, seq in cytc_sequences.items():
    print(f"  {name:>14}: {len(seq)} residues")
print(f"\nFASTA written to: {cytc_fasta}")

In [ ]:
# Try to align with an available tool, fall back to built-in method
aligned_cytc_path = os.path.join(tempfile.gettempdir(), 'cytc_aligned.fasta')
aligned = False

for tool in ['muscle', 'mafft', 'clustalo']:
    if run_msa_tool(tool, cytc_fasta, aligned_cytc_path):
        aligned = True
        break

if aligned and os.path.exists(aligned_cytc_path):
    aln_cytc = AlignIO.read(aligned_cytc_path, 'fasta')
    print(f"\nAlignment: {len(aln_cytc)} sequences, {aln_cytc.get_alignment_length()} columns")
    print()
    for record in aln_cytc:
        print(f"{record.id:>14}  {str(record.seq)}")
else:
    print("\nNo MSA tool available. Creating manual alignment for demonstration:")
    # Pre-computed alignment for fallback
    cytc_aligned = [
        SeqRecord(Seq("M-GDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE"), id="CYC_HUMAN"),
        SeqRecord(Seq("M-GDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKGERADLIAYLKKATNE"), id="CYC_HORSE"),
        SeqRecord(Seq("M-GDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKSEREDLIAYLKKATNE"), id="CYC_MOUSE"),
        SeqRecord(Seq("M-GDIEKGKKIFVQKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAEGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKSERVDLIAYLKDATSK"), id="CYC_CHICKEN"),
        SeqRecord(Seq("M-GDVAKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLIGRKTGQAEGFSYTDANKSKGIVWNENILMEYLENPKKYIPGTKMIFAGIKKKGERQDLIAYLKSATS-"), id="CYC_TUNA"),
        SeqRecord(Seq("MTEFKAGSAKKGATLFKTRCLQCHTVEKGGPHKVGPNLHGIFGRHSGQAEGYSYTDANIKKNVLWDENNMSEYLTNPKKYIPGTKMAFGGLKKEKDRNDLITYLKKACE"), id="CYC_YEAST"),
    ]
    aln_cytc = MultipleSeqAlignment(cytc_aligned)
    for record in aln_cytc:
        print(f"{record.id:>14}  {str(record.seq)}")

In [ ]:
# Analyze the cytochrome c alignment
cytc_aln_strs = [str(record.seq) for record in aln_cytc]
cytc_names = [record.id for record in aln_cytc]

# Conservation analysis
cytc_consensus, cytc_cons = compute_consensus(cytc_aln_strs, threshold=0.6)

print("Cytochrome c alignment analysis:")
print(f"  Sequences: {len(cytc_aln_strs)}")
print(f"  Alignment length: {len(cytc_aln_strs[0])}")
print(f"  Fully conserved positions: {sum(1 for s in cytc_cons if s >= 1.0)}")
print(f"  Highly conserved (>=80%): {sum(1 for s in cytc_cons if s >= 0.8)}")
print(f"  Mean conservation: {np.mean(cytc_cons):.3f}")
print(f"\n  Consensus (60%): {cytc_consensus}")

# Find conserved blocks
cytc_blocks, cytc_full_cons = find_conserved_blocks(cytc_aln_strs, min_conservation=0.8, min_length=4)
print(f"\n  Conserved blocks (>= 80%, >= 4 residues): {len(cytc_blocks)}")
for start, end, mc in cytc_blocks:
    print(f"    [{start}-{end}] length={end-start}, conservation={mc:.2f}")

In [ ]:
# Visualize the alignment
plot_alignment_heatmap(cytc_aln_strs, cytc_names, 
                       title="Cytochrome c Multiple Sequence Alignment")

In [ ]:
# Pairwise identity matrix
plot_identity_matrix(cytc_aln_strs, cytc_names)

In [ ]:
# Information content / sequence logo
cytc_pos, cytc_ic = information_content(cytc_aln_strs, alphabet_size=20)
plot_sequence_logo(cytc_pos, cytc_ic, start=0, end=50,
                   title="Cytochrome c - Sequence Logo (positions 0-50)")

---

## 12. Exercises

### Exercise 1: Align a Protein Family

Given the following kinase sequences, perform an MSA and identify the conserved catalytic residues.

In [ ]:
# Exercise 1: Align these kinase-like sequences
kinase_seqs = {
    'ABL1_HUMAN':  'MELPIDPEGSELEVEFLNQTLDEFGKVYGTLWKDNSLGQRLR',
    'SRC_HUMAN':   'MELRIDDEGTLEVETFNQTTLDEFGKVYGTLWKENNLGQRIR',
    'EGFR_HUMAN':  'MELPSDKGSQVEKVFLNQTLDEFGKVYGTIWKENSLGQRIR',
    'BRAF_HUMAN':  'MELPTDKGAQVEKIFLNQTLREFGKVYGTLWKETKLGQRLR',
    'CDK2_HUMAN':  'MELPLDKGGQVEKVYLNQTLHEFGKVYGTLWKETHLGQRIR',
}

# YOUR CODE HERE:
# 1. Write sequences to a FASTA file
# 2. Align using an available tool (or simple_nw_align for pairs)
# 3. Calculate conservation scores
# 4. Identify the most conserved positions
# 5. What residues are invariant across all kinases?

print("Exercise 1: Align these kinase sequences and find conserved catalytic residues.")
print("Hint: The GXGXXG motif (glycine-rich loop) is key for ATP binding.")

### Exercise 2: Compare MSA Scoring Methods

Create two different alignments of the same sequences (e.g., by shuffling the guide tree order) and compare their Sum-of-Pairs scores.

In [ ]:
# Exercise 2: Compare two MSA scorings

# Alignment A (a "good" alignment)
alignment_A = [
    "MVLSPADKTNVKA",
    "MVLSGEDKSNIKA",
    "MVHLTPEEKSAVT",
]

# Alignment B (a "bad" alignment - shifted)
alignment_B = [
    "MVLSPADKTNVKA",
    "-MVLSGEDKSNIK",
    "MVHLTPEEKSAV-",
]

# YOUR CODE HERE:
# 1. Calculate SP score for both alignments
# 2. Calculate per-column conservation for both
# 3. Which alignment is better and why?
# 4. Plot the column-by-column comparison

print("Exercise 2: Compare the SP scores of alignments A and B.")
print("Which alignment better captures the biological signal?")

### Exercise 3: Build a Consensus Sequence

From the cytochrome c alignment, build three types of consensus and determine which invariant residues are likely functionally essential.

In [ ]:
# Exercise 3: Consensus analysis

# YOUR CODE HERE:
# 1. Compute strict consensus (100% threshold)
# 2. Compute majority consensus (50% threshold)
# 3. List all fully invariant positions and their amino acids
# 4. Research: which of these are known to be functionally important?
#    Hint: Cysteine residues in cytochrome c coordinate the heme group

print("Exercise 3: Build consensus sequences and identify functionally important residues.")
print("Hint: Look for conserved C, H, and K residues.")

### Exercise 4: Information Content Analysis

Compute the information content for a DNA alignment and compare it to a protein alignment.

In [ ]:
# Exercise 4: Information content for DNA vs protein

dna_alignment = [
    "ATGATGCTGAGCCCAGCAG",
    "ATGATGCTGAGCCCAGCAG",
    "ATGATGTTGAGCCCAGCAG",
    "ATGATGCTAAGCCCAGCAG",
    "ATGATGCTGAGCCCAGCAG",
    "ATGATGCTGAGTCCAGCAG",
]

protein_alignment = [
    "MMLSPAQ",
    "MMLSPAQ",
    "MMLSPAQ",
    "MMLSPAQ",
    "MMLSPAQ",
    "MMLSPAQ",
]

# YOUR CODE HERE:
# 1. Calculate IC for DNA alignment (alphabet_size=4)
# 2. Calculate IC for protein alignment (alphabet_size=20)
# 3. Compare: which has higher maximum possible IC? Why?
# 4. Plot both IC profiles side by side

print("Exercise 4: Compare information content between DNA and protein alignments.")
print(f"Max IC for DNA: {math.log2(4):.2f} bits")
print(f"Max IC for protein: {math.log2(20):.2f} bits")

### Exercise 5: Motif Discovery from MSA

Given an alignment of SH3 domains, find the conserved binding motif.

In [ ]:
# Exercise 5: Find conserved motifs in SH3 domains

sh3_alignment = [
    "AEEIYDRQGFDAYINLHPEDLNDNLTA-LPYWDQDLPKEGEAPQLALIVDR",
    "AEEIYEQQGFDAYINLHPEDLNDNLTA-LPYWDQELPREGEAPQLALIVDR",
    "ADEIYEQQGFDAYINLHPDDLNDNLTA-LPYWDQKLPKEGEAPQLALIVDR",
    "AEEIYDRQGFDAYINLHPEDLNDNLTA-LPYWDQDLPKEGEAPQLALIVDR",
    "AEALYDRQGFDAYINLHPEDLNDNLSA-LPYWDQDLPKEGEAPQLALIVDR",
    "ADEIYEQQGFDSYINLHPDDLNDNLTA-LPYWDQELPKEGEAPQLALIVDR",
]

# YOUR CODE HERE:
# 1. Calculate per-position conservation
# 2. Find all conserved blocks (>= 80% conservation, length >= 3)
# 3. Extract the consensus motif for each block
# 4. Which motifs correspond to known SH3 functional regions?
# 5. Plot the conservation profile

print("Exercise 5: Find conserved motifs in SH3 domain alignment.")
print("Hint: SH3 domains have a characteristic beta-barrel fold with conserved aromatic residues.")

### Exercise 6: Progressive Alignment Sensitivity

Demonstrate how different guide trees lead to different alignments.

In [ ]:
# Exercise 6: Guide tree sensitivity

test_sequences = [
    "ATCGATCGATCG",
    "ATCGTTCGATCG",
    "TTCGATCGATCG",
    "TTCGTTCGATCG",
    "AACCGGTTAACC",
]

# YOUR CODE HERE:
# 1. Compute k-mer distance matrix
# 2. Build UPGMA guide tree
# 3. Manually try aligning in a different order:
#    a) Start with sequences 0 and 4 (most distant pair)
#    b) Start with sequences 0 and 1 (closest pair)
# 4. Compare the resulting MSAs using SP score
# 5. Which order gives a better result? Why?

print("Exercise 6: Show that guide tree order matters for progressive MSA.")

---

## Summary

### Key Concepts

| Concept | Description |
|---------|-------------|
| **MSA** | Simultaneous alignment of 3+ sequences revealing conservation |
| **Progressive alignment** | Heuristic: build guide tree, align pairs/groups iteratively |
| **Guide tree** | Rough phylogeny (UPGMA/NJ) to order alignment steps |
| **Profile alignment** | Aligning groups using averaged substitution scores |
| **Sum-of-pairs score** | MSA quality: sum of all pairwise scores per column |
| **Conservation score** | Per-position measure of sequence similarity |
| **Information content** | Bits of information at each position (entropy-based) |
| **Sequence logo** | Visual representation of position-specific conservation |
| **Conserved blocks** | Contiguous regions of high conservation |

### Tools Comparison

- **ClustalW/Omega**: Good default; Omega scales to large datasets via mBed clustering
- **MUSCLE**: Fast iterative refinement; good balance of speed and accuracy
- **MAFFT**: Excellent for large/diverse datasets; FFT-based anchor finding
- **T-Coffee**: Best accuracy for small datasets; consistency-based scoring

### Best Practices

1. Start with MAFFT `--auto` for most tasks
2. Use T-Coffee or PROBCONS for small, critical alignments
3. Always inspect alignments visually (JalView or programmatically)
4. Trim poorly aligned regions before downstream analysis
5. Consider iterative refinement for improved accuracy

---

## Resources

- [Clustal Omega](https://www.ebi.ac.uk/Tools/msa/clustalo/) -- EBI web server
- [MUSCLE](https://www.drive5.com/muscle/) -- Robert Edgar's tool
- [MAFFT](https://mafft.cbrc.jp/alignment/software/) -- Multiple alignment with FFT
- [T-Coffee](http://tcoffee.crg.cat/) -- Consistency-based MSA
- [JalView](https://www.jalview.org/) -- Interactive alignment viewer
- [BioPython AlignIO docs](https://biopython.org/wiki/AlignIO) -- File format handling
- [WebLogo](https://weblogo.berkeley.edu/) -- Online sequence logo generator

---

[< Previous: BLAST: Sequence Similarity Searching](../04_BLAST_Searching/01_blast_searching.ipynb) | [Tier 2: Core Bioinformatics Overview](../README.md) | [Next: Phylogenetics: Reconstructing Evolutionary History >](../06_Phylogenetics/01_phylogenetics.ipynb)

---